In [ ]:
# 确认 GPU
!nvidia-smi

In [ ]:
# 挂载 Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 创建持久化目录
!mkdir -p "/content/drive/MyDrive/MuVisual/input"
!mkdir -p "/content/drive/MyDrive/MuVisual/output"
!mkdir -p "/content/drive/MyDrive/MuVisual/cache/huggingface"
!mkdir -p "/content/drive/MyDrive/MuVisual/cache/torch"
!mkdir -p "/content/drive/MyDrive/MuVisual/cache/separator"
# 后续将待处理音频上传到：MyDrive/MuVisual/input

In [ ]:
# 下载项目
%cd /content
!git clone https://github.com/TecReaGroup/MuVisual-Workflow.git
%cd /content/MuVisual-Workflow
# 更新已有代码
# %cd /content/MuVisual-Workflow
# !git pull

In [ ]:
# 安装系统依赖
!apt-get update -qq
!apt-get install -y -qq ffmpeg libsndfile1 build-essential
!ffmpeg -version | head -n 1

In [ ]:
# 安装 uv 和项目环境
!curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
# 添加 PATH，并安装项目锁定的 Python 3.12 和全部依赖
import os

os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
%cd /content/MuVisual-Workflow
!uv python install 3.12
!uv sync --locked --python 3.12

In [ ]:
from google.colab import userdata
import os

token = userdata.get("HF_TOKEN")
assert token and token.startswith("hf_"), "Colab Secret HF_TOKEN 不存在或格式错误"

os.environ["HF_TOKEN"] = token
os.environ["HF_HOME"] = "/content/drive/MyDrive/MuVisual/cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/content/drive/MyDrive/MuVisual/cache/huggingface/hub"

print("HF_TOKEN 已传入当前 Colab 环境")

In [ ]:
# 设置持久化模型缓存
import os

drive_root = "/content/drive/MyDrive/MuVisual"

os.environ["HF_HOME"] = f"{drive_root}/cache/huggingface"
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{drive_root}/cache/huggingface/hub"
os.environ["TORCH_HOME"] = f"{drive_root}/cache/torch"
os.environ["AUDIO_SEPARATOR_MODEL_DIR"] = f"{drive_root}/cache/separator"

In [ ]:
# 检查 Torch 和 CUDA
%cd /content/MuVisual-Workflow
!uv run python -c "import torch; print('Torch:', torch.__version__); print('CUDA wheel:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')"

In [ ]:
# 执行批量处理
%cd /content/MuVisual-Workflow

!uv run muvisual \
  --input "/content/drive/MyDrive/MuVisual/input" \
  --output "/content/drive/MyDrive/MuVisual/output" \
  --device cuda